# 13. PQ — 벡터를 바이트 몇 개로, 거리는 표 조회로

슬라이드 「PQ 원리」 · 「PQ - ADC」 · 「IVF-PQ」
1. D차원 벡터를 **m 조각**으로 자른다 (64차원 → 8조각 × 8차원)
2. 조각마다 k-means 로 **256개 대표점(코드북)** → 조각 하나 = 1바이트 코드
3. 질의는 압축하지 않고, 조각별로 256개 대표점과의 거리를 **표(LUT)** 로 미리 계산
4. 레코드와의 거리 = 표에서 m 번 꺼내 더하기

> 주의: 결과는 "8차원 벡터"가 아니라 **8바이트 코드**다. 차원 축소(PCA)와 다르다.

In [ ]:
from labutil import *


def sqdist(A, B):
    return (A ** 2).sum(1)[:, None] - 2 * A @ B.T + (B ** 2).sum(1)[None, :]


def kmeans(X, k, iters=20, seed=0):
    rng = np.random.default_rng(seed)
    C = X[rng.choice(len(X), k, replace=False)].copy()
    for _ in range(iters):
        lab = sqdist(X, C).argmin(1)
        cnt = np.bincount(lab, minlength=k)
        S = np.zeros_like(C); np.add.at(S, lab, X)
        C = np.where(cnt[:, None] > 0, S / np.maximum(cnt, 1)[:, None], C)
    return C


N, D = 20_000, 64
X, _ = make_clusters(N + 200, D, n_clusters=60, center_scale=0.8, seed=21)
X, Q = X[:N], X[N:]
truth = brute_knn(X, Q, 10)

## 1. 자르고, 조각마다 코드북을 만든다

In [ ]:
class PQ:
    def __init__(self, m=8, ks=256):
        self.m, self.ks = m, ks

    def fit(self, X):
        self.ds = X.shape[1] // self.m                        # 조각 하나의 차원 (64/8 = 8)
        self.books = [kmeans(X[:, j * self.ds:(j + 1) * self.ds], self.ks, seed=j) for j in range(self.m)]
        return self

    def encode(self, X):                                      # 조각마다 가장 가까운 대표점 번호 → uint8
        return np.stack([sqdist(X[:, j * self.ds:(j + 1) * self.ds], B).argmin(1)
                         for j, B in enumerate(self.books)], axis=1).astype(np.uint8)

    def decode(self, codes):                                  # 코드 → 대표점을 이어 붙인 근사 벡터
        return np.hstack([self.books[j][codes[:, j]] for j in range(self.m)])

    def lut(self, qv):                                        # 질의 조각 ↔ 256 대표점 거리² 표 (m × 256)
        return np.stack([sqdist(qv[None, j * self.ds:(j + 1) * self.ds], B)[0] for j, B in enumerate(self.books)])

    def adc(self, qv, codes):                                 # 표 조회 m 번의 합
        T = self.lut(qv)
        return T[np.arange(self.m), codes].sum(1)


with timer("코드북 학습 (k-means 8번)"):
    pq = PQ(m=8).fit(X)
codes = pq.encode(X)
print("코드 모양:", codes.shape, codes.dtype, "  첫 레코드의 코드:", codes[0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 3.6), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]
ax.imshow(X[:6], aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
for j in range(1, 8):
    ax.axvline(j * 8 - 0.5, color="black", lw=2)
for r in range(6):
    for j in range(8):
        ax.text(j * 8 + 3.5, r, str(codes[r, j]), ha="center", va="center", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
ax.set_yticks(range(6)); ax.set_ylabel("레코드"); ax.set_xlabel("64개 차원 (8개씩 8조각)")
ax.set_title("벡터 6개 → 조각마다 가장 가까운 대표점 번호(0~255) = 8바이트")
ax = axes[1]
sub = X[:3000, :2]
ax.scatter(*sub.T, s=2, color="#9aa5a8", label="데이터 (조각 1의 앞 2차원)")
ax.scatter(*pq.books[0][:, :2].T, s=12, color="#FF7043", label="대표점 256개")
ax.legend(fontsize=8, loc="lower left"); ax.set_title("조각 1의 코드북")
plt.tight_layout(); plt.show()

In [ ]:
float_bytes, code_bytes = D * 4, pq.m
book_bytes = pq.m * pq.ks * pq.ds * 4
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.barh(["원본 float32", "PQ 코드"], [N * float_bytes / 2**20, N * code_bytes / 2**20], color=["#3D484A", "#FF7043"])
for i, v in enumerate([N * float_bytes / 2**20, N * code_bytes / 2**20]):
    ax.text(v, i, f" {v:.2f} MB", va="center")
ax.set_xlabel("MB (2만 개)"); ax.set_title(f"레코드당 {float_bytes}B → {code_bytes}B ({float_bytes // code_bytes}배 압축) + 코드북 {book_bytes / 1024:.0f}KB 한 번")
plt.tight_layout(); plt.show()

## 2. ADC — 질의 하나로 표를 만들고, 더하기만 한다

In [ ]:
qv = Q[0]
T = pq.lut(qv)
r = 0
parts = T[np.arange(8), codes[r]]

fig, axes = plt.subplots(1, 2, figsize=(15, 3.4), gridspec_kw={"width_ratios": [2.2, 1]})
im = axes[0].imshow(T, aspect="auto", cmap="viridis")
axes[0].scatter(codes[r], range(8), marker="s", s=60, facecolors="none", edgecolors="#FF7043", lw=2)
axes[0].set_xlabel("대표점 번호 0~255"); axes[0].set_ylabel("조각"); plt.colorbar(im, ax=axes[0], label="거리²")
axes[0].set_title("질의의 거리표 LUT (8 × 256) · □ = 레코드 0 의 코드가 가리키는 칸")
axes[1].bar(range(1, 9), parts, color="#009CA6")
axes[1].set_xlabel("조각"); axes[1].set_title(f"8칸을 더하면 {parts.sum():.2f}")
plt.tight_layout(); plt.show()

x_hat = pq.decode(codes[r:r + 1])[0]
print(f"표 조회 합                ‖q − x̂‖² = {parts.sum():.4f}")
print(f"복원 벡터로 직접 계산      ‖q − x̂‖² = {((qv - x_hat) ** 2).sum():.4f}   ← 똑같다 (조각별 합 = 전체)")
print(f"원본 벡터와의 실제 거리    ‖q − x‖²  = {((qv - X[r]) ** 2).sum():.4f}   ← 이것과의 차이가 압축 손실")

**관찰:** 거리² 는 차원별 제곱의 합이라 조각별 합으로 정확히 나뉜다 (슬라이드: "교차항이 없다").
그래서 표 조회의 합은 **복원 벡터 x̂ 와의 거리**와 똑같다. 원본 x 와의 차이만 손실이다.

## 3. 근사 거리는 얼마나 정확한가

In [ ]:
true_d = ((X - qv) ** 2).sum(1)
adc_d = pq.adc(qv, codes)
top = np.argsort(true_d)[:200]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
axes[0].scatter(true_d, adc_d, s=2, alpha=0.3, color="#3D484A", label="전체 2만 개")
axes[0].scatter(true_d[top], adc_d[top], s=6, color="#FF7043", label="진짜 가까운 200개")
lim = [0, true_d.max()]; axes[0].plot(lim, lim, "--", color="gray")
axes[0].set_xlabel("실제 거리²"); axes[0].set_ylabel("ADC 추정 거리²"); axes[0].legend()
axes[0].set_title(f"상관계수 {np.corrcoef(true_d, adc_d)[0, 1]:.3f}")
rank_adc = np.argsort(np.argsort(adc_d))
axes[1].scatter(range(1, 51), rank_adc[np.argsort(true_d)[:50]] + 1, color="#009CA6")
axes[1].plot([1, 50], [1, 50], "--", color="gray")
axes[1].set_xlabel("실제 순위"); axes[1].set_ylabel("ADC 순위"); axes[1].set_title("상위 50개의 순위가 뒤섞인다")
plt.tight_layout(); plt.show()

**관찰:** 전체적으로는 잘 맞지만 가까운 쪽에서 순위가 뒤섞인다. 손실 압축이라 recall 이 떨어진다.
해결책: PQ 로 후보를 넉넉히 뽑고 **원본 벡터로 재정렬** (슬라이드 "압축본 회수 → 원본 재정렬").

In [ ]:
def pq_search(qv, k=10, rerank=0, pq=pq, codes=codes):
    d = pq.adc(qv, codes)
    if not rerank:
        return np.argsort(d)[:k]
    cand = np.argpartition(d, rerank)[:rerank]              # PQ 로 후보 rerank 개
    exact = ((X[cand] - qv) ** 2).sum(1)                     # 원본으로 다시 계산
    return cand[np.argsort(exact)[:k]]


rr = [0, 20, 50, 100, 200, 500]
rec = [recall([pq_search(qq, rerank=r) for qq in Q], truth) for r in rr]

ms = {}
for m in [4, 8, 16, 32]:
    p = PQ(m=m).fit(X)
    c = p.encode(X)
    ms[m] = (recall([pq_search(qq, pq=p, codes=c) for qq in Q], truth),
             recall([pq_search(qq, rerank=100, pq=p, codes=c) for qq in Q], truth))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot([str(r) if r else "없음" for r in rr], rec, "-o", color="#FF7043")
for i, v in enumerate(rec):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center")
axes[0].set_ylim(0, 1.08); axes[0].set_xlabel("원본으로 재정렬할 후보 수"); axes[0].set_ylabel("recall@10")
axes[0].set_title("PQ(m=8) 후보 → 원본 재정렬")
xs = np.arange(len(ms))
axes[1].bar(xs - 0.2, [v[0] for v in ms.values()], 0.4, label="PQ 만", color="#9aa5a8")
axes[1].bar(xs + 0.2, [v[1] for v in ms.values()], 0.4, label="PQ + 재정렬 100", color="#009CA6")
axes[1].set_xticks(xs, [f"m={m}\n{m}B/벡터" for m in ms]); axes[1].set_ylim(0, 1.05); axes[1].legend()
axes[1].set_title("조각 수 m ↑ → 코드 커짐 · recall ↑")
plt.tight_layout(); plt.show()

## 4. IVF-PQ — 잔차(residual)를 압축하면 더 정확하다
슬라이드: "정밀도를 위해 잔차(벡터 − 중심)를 PQ 로 압축하는 방식이 표준"
벡터 자체보다 **중심에서 얼마나 벗어났나**는 값의 범위가 작아서, 같은 256개 대표점으로 더 촘촘히 표현된다.

In [ ]:
nlist = 141
C = kmeans(X, nlist, seed=0)
lab = sqdist(X, C).argmin(1)
R = X - C[lab]                                               # 잔차

pq_raw, pq_res = PQ(m=8).fit(X), PQ(m=8).fit(R)
err_raw = ((X - pq_raw.decode(pq_raw.encode(X))) ** 2).sum(1).mean()
err_res = ((R - pq_res.decode(pq_res.encode(R))) ** 2).sum(1).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].hist(X[:, 0], bins=60, alpha=0.6, label="원본 벡터의 1번 차원", color="#3D484A")
axes[0].hist(R[:, 0], bins=60, alpha=0.6, label="잔차의 1번 차원", color="#FF7043")
axes[0].legend(); axes[0].set_title("잔차는 0 근처에 모인다")
axes[1].bar(["원본을 PQ", "잔차를 PQ"], [err_raw, err_res], color=["#3D484A", "#FF7043"])
for i, v in enumerate([err_raw, err_res]):
    axes[1].text(i, v, f"{v:.1f}", ha="center", va="bottom")
axes[1].set_title("압축 오차 (복원 벡터와의 평균 거리²) — 같은 8바이트")
plt.tight_layout(); plt.show()

**관찰:**
- 같은 8바이트로 잔차를 압축하면 오차가 줄어든다. 대신 중심 번호(IVF 리스트)를 알아야 복원할 수 있다
- 그래서 IVF-PQ 는 "IVF 로 뒤질 군집을 고르고 → 그 군집의 잔차 코드를 LUT 로 비교 → 원본 재정렬" 순서다
- **pgvector 에는 PQ 가 없다.** 대신 halfvec(절반 크기) · 이진 양자화(32배) + 재정렬을 쓴다 → 23번